<a href="https://colab.research.google.com/github/TajbeerAhamed/Data-Mining-Lab-Assignments/blob/main/Project_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import networkx as nx
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

Stopwords are used when building the inverted index. The inverted index will ignore stopwords.

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

STOPWORDS = stopwords.words('english')
print(STOPWORDS)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Add custom stopwords if you deem it necessary

In [ ]:
custom_STOPWORDS = [] # Add your own stopwords here
STOPWORDS.extend(custom_STOPWORDS)

In [ ]:
from collections import defaultdict

# Inverted index: word -> set of URLs
inverted_index = defaultdict(set)
url_list = set()

In [ ]:
# This dictionary will be used to build the connection between links
web_connection = {'source':[], 'target':[]}

In [ ]:
import re

# This function will clean the content of web page in order to build the inverted index.
def clean_and_tokenize(text):
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text.lower())  # Remove punctuation and lowercase
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

In [ ]:
from urllib.parse import urljoin, urlparse

# The crawl function has 5 parameters
# url = The url to crawl
# base_domain = the base domain of the url. During crawling, the crawler will ignore links from other domains

def crawl(url, base_domain, visited, visit_limit, limit):
    if limit==0 or len(visited)==visit_limit:
        return

    try:
        response = requests.get(url, timeout=5)
        if response.status_code != 200:
            return
    except requests.RequestException:
        return

    visited.add(url)
    print("-"*(10-limit), end=" ")
    print(f"Crawled: {url}")

    soup = BeautifulSoup(response.text, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)
    words = clean_and_tokenize(text)

    for word in words:
        inverted_index[word].add(url)
        url_list.add(url)

    # Recursively follow links
    for tag in soup.find_all('a', href=True):
        link = urljoin(url, tag['href'])
        parsed = urlparse(link)

        # Store external links as connection
        web_connection['source'].append(url)
        web_connection['target'].append(link)

        if parsed.netloc == base_domain and link not in visited:
            crawl(link, base_domain, visited, visit_limit, limit-1)

In [ ]:
def crawl_roots(root_urls, max_per_root=3, visit_limit=15):
    for root in root_urls:
        print(f"\nStarting crawl from: {root}")
        domain = urlparse(root).netloc
        visited = set()
        crawl(root, domain, visited, visit_limit, max_per_root)

In [ ]:
seed_urls = [
    'https://www.lonelyplanet.com',

'https://www.tripadvisor.com',

'https://www.booking.com',

'https://www.expedia.com',

'https://www.airbnb.com',

'https://www.kayak.com',

'https://www.skyscanner.com',

'https://www.nomadicmatt.com',

'https://www.theplanetd.com',

'https://www.atlasobscura.com',


]

crawl_roots(seed_urls, max_per_root=10)


Starting crawl from: https://www.lonelyplanet.com
 Crawled: https://www.lonelyplanet.com
- Crawled: https://www.lonelyplanet.com/
-- Crawled: https://www.lonelyplanet.com/profile/saves
--- Crawled: https://www.lonelyplanet.com/legal/website-terms
---- Crawled: https://www.lonelyplanet.com/legal/website-terms#definitions
----- Crawled: https://www.lonelyplanet.com/legal/website-terms#using
------ Crawled: https://www.lonelyplanet.com/legal/website-terms#prohibited
------- Crawled: https://www.lonelyplanet.com/legal/website-terms#trademarks
-------- Crawled: https://www.lonelyplanet.com/legal/website-terms#contributing
--------- Crawled: https://www.lonelyplanet.com/legal/website-terms#removal
--------- Crawled: https://www.lonelyplanet.com/legal/website-terms#abuse
--------- Crawled: https://www.lonelyplanet.com/legal/website-terms#defamation
--------- Crawled: https://www.lonelyplanet.com/legal/website-terms#copyright
--------- Crawled: https://www.lonelyplanet.com/legal/website-terms

In [ ]:
# print inverted index
print("\nSample inverted index (first 20 words):")
for word in list(inverted_index.keys())[:20]:
    print(f"{word}: {list(inverted_index[word])}")


Sample inverted index (first 20 words):
lonely: ['https://www.lonelyplanet.com/legal/website-terms#abuse', 'https://www.lonelyplanet.com/legal/website-terms#trademarks', 'https://www.lonelyplanet.com', 'https://www.lonelyplanet.com/', 'https://www.lonelyplanet.com/legal/website-terms#using', 'https://www.lonelyplanet.com/legal/website-terms#copyright', 'https://www.lonelyplanet.com/legal/website-terms#contributing', 'https://www.lonelyplanet.com/profile/saves', 'https://www.lonelyplanet.com/legal/website-terms#definitions', 'https://www.lonelyplanet.com/legal/website-terms#defamation', 'https://www.lonelyplanet.com/legal/website-terms', 'https://www.lonelyplanet.com/legal/website-terms#prohibited', 'https://www.lonelyplanet.com/legal/website-terms#liability', 'https://www.lonelyplanet.com/legal/website-terms#privacy', 'https://www.lonelyplanet.com/legal/website-terms#removal']
planet: ['https://www.lonelyplanet.com/legal/website-terms#using', 'https://www.lonelyplanet.com/', 'https://

In [ ]:
# Print first 20 connections

for source, target in list(zip(web_connection['source'], web_connection['target']))[:20]:
    print(f"{source} -> {target}")

https://www.lonelyplanet.com -> https://www.lonelyplanet.com/
https://www.lonelyplanet.com/ -> https://www.lonelyplanet.com/
https://www.lonelyplanet.com/ -> https://www.lonelyplanet.com/profile/saves
https://www.lonelyplanet.com/profile/saves -> https://www.lonelyplanet.com/legal/website-terms
https://www.lonelyplanet.com/legal/website-terms -> https://www.lonelyplanet.com/
https://www.lonelyplanet.com/legal/website-terms -> https://www.lonelyplanet.com/profile/saves
https://www.lonelyplanet.com/legal/website-terms -> https://www.lonelyplanet.com/legal/website-terms#definitions
https://www.lonelyplanet.com/legal/website-terms#definitions -> https://www.lonelyplanet.com/
https://www.lonelyplanet.com/legal/website-terms#definitions -> https://www.lonelyplanet.com/profile/saves
https://www.lonelyplanet.com/legal/website-terms#definitions -> https://www.lonelyplanet.com/legal/website-terms#definitions
https://www.lonelyplanet.com/legal/website-terms#definitions -> https://www.lonelyplanet

In [ ]:
web_graph = nx.DiGraph()
for source, target in zip(web_connection['source'], web_connection['target']):
  web_graph.add_edge(source, target) # Iterate through source and target lists using zip

In [ ]:
pagerank_scores = nx.pagerank(web_graph, alpha=0.85, max_iter=100, tol=1e-6)
print("\nPageRank Scores:", pagerank_scores)


PageRank Scores: {'https://www.lonelyplanet.com': 0.00027178155394009477, 'https://www.lonelyplanet.com/': 0.00032011202759263304, 'https://www.lonelyplanet.com/profile/saves': 0.00032011202759263304, 'https://www.lonelyplanet.com/legal/website-terms': 0.00041063480157112707, 'https://www.lonelyplanet.com/legal/website-terms#definitions': 0.00031439996154594224, 'https://www.lonelyplanet.com/legal/website-terms#using': 0.00031439996154594224, 'https://www.lonelyplanet.com/legal/website-terms#prohibited': 0.00031439996154594224, 'https://www.lonelyplanet.com/legal/website-terms#trademarks': 0.00031439996154594224, 'https://www.lonelyplanet.com/legal/website-terms#contributing': 0.00031439996154594224, 'https://www.lonelyplanet.com/legal/website-terms#removal': 0.00031439996154594224, 'https://www.lonelyplanet.com/legal/website-terms#abuse': 0.00031439996154594224, 'https://www.lonelyplanet.com/legal/website-terms#defamation': 0.00031439996154594224, 'https://www.lonelyplanet.com/legal/

In [ ]:
def search_engine(query, index, scores):
    query_terms = query.lower().split()
    results = set()
    for term in query_terms:
        if term in index:
            if not results:
                results = set(index[term])
            else:
                results = results.intersection(index[term])  # Find common websites

    # Sort results based on score
    ranked_results = []
    for website in results:
        if website in scores:
          ranked_results.append((website, scores[website]))
    ranked_results.sort(key=lambda x: x[1], reverse=True)

    return ranked_results

In [ ]:
# Query and display results
query = "sea"
print(f"\nSearch Results for '{query}' using PageRank:")
results = search_engine(query, inverted_index, pagerank_scores)

for page, score in results:
    print(f"{page}: ({score})")


Search Results for 'sea' using PageRank:
https://www.expedia.com/Cruises: (0.00039848981874762143)
https://www.atlasobscura.com/: (0.00033676227938884626)
https://www.nomadicmatt.com/travel-blogs/the-ultimate-guide-to-traveling-when-you-have-no-money/: (0.00032263059367400624)
https://www.kayak.com/: (0.0002975347307058188)
https://www.kayak.com/flights: (0.0002975347307058188)
https://www.kayak.com/packages/honolulu: (0.00028518559400641006)
https://www.kayak.com/stays: (0.0002828328811095797)
https://www.booking.com/packages.en-gb.html: (0.0002787251855057278)
https://www.booking.com/articles/best-ryokans-japan.en.html: (0.0002760476257838468)
https://www.booking.com/packages.en-gb.html#indexsearch: (0.0002745007474732599)
https://www.kayak.com/flights#main: (0.00027289918650694005)
https://www.kayak.com/#main: (0.0002727533543967255)
https://www.kayak.com: (0.0002727090893362952)
https://www.kayak.com#main: (0.0002727090893362952)
https://www.atlasobscura.com: (0.000271781553940094